# Ames Housing — Simple Week 1 Data Cleaning (Regression)

Download **Kaggle House Prices `train.csv`** and put it beside this notebook. Run every cell from top to bottom.

This notebook uses just **pandas** and **NumPy**. It does not download anything, use `pathlib`, generate a Markdown report automatically, train a model, encode predictors, or scale values.

**Important:** We treat `"NA"` and empty cells as missing, but preserve the literal text `"None"` because it can mean that a house has no masonry veneer. `SalePrice` is the target variable and must remain unchanged.

After running the notebook, submit the exported CSV reports, cleaned CSV, and the separate `DATA_QUALITY_REPORT.md` (edit the brief results section using your notebook output).

## 1. Import libraries and load Kaggle train.csv

In [ ]:
import pandas as pd

# Put train.csv in the same folder as this notebook.
# Keep real text "None"; treat Kaggle's "NA" as missing.
raw = pd.read_csv("train.csv", keep_default_na=False, na_values=["NA", ""])
df = raw.copy()

# Remember the target and original row count so we can verify them later.
original_price = raw["SalePrice"].copy()
original_ids = raw["Id"].copy()

print("Dataset size:", raw.shape)
print("First five rows:")
print(raw.head())

## 2. Full missing-value report

In [ ]:
missing = pd.DataFrame({
    "Column": raw.columns,
    "Missing_Count": raw.isna().sum().values
})
missing["Missing_Percent"] = (missing["Missing_Count"] / len(raw) * 100).round(2)
missing = missing.sort_values("Missing_Count", ascending=False)

print(missing.to_string(index=False))
print("Columns with missing values:", (missing["Missing_Count"] > 0).sum())
print("Total missing cells:", raw.isna().sum().sum())

missing.to_csv("missing_values_report.csv", index=False)

## 3. Find exact and possible near duplicates

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicates ignoring Id:", raw.drop(columns="Id").duplicated().sum())
print("Repeated Id values:", raw["Id"].duplicated().sum())

# Similar measurements can belong to different houses: review, do not delete.
check_columns = ["Neighborhood", "YearBuilt", "OverallQual", "GrLivArea"]
near_duplicates = raw[raw.duplicated(subset=check_columns, keep=False)]
print("Possible near-duplicate rows:", len(near_duplicates))
print(near_duplicates[["Id"] + check_columns].head(10))
near_duplicates.to_csv("near_duplicate_candidates.csv", index=False)

## 4. Fix meaningful missing categories (no feature exists)

In [ ]:
# NA means a feature does not exist for these fields.
for column in ["Alley", "Fence", "MiscFeature"]:
    df[column] = df[column].fillna("None")

# Use another column to check if the feature is truly absent.
# If the feature exists but quality is missing, say "Unknown" instead.
feature_columns = {
    "PoolQC": "PoolArea",
    "FireplaceQu": "Fireplaces",
    "GarageType": "GarageArea",
    "GarageFinish": "GarageArea",
    "GarageQual": "GarageArea",
    "GarageCond": "GarageArea",
    "BsmtQual": "TotalBsmtSF",
    "BsmtCond": "TotalBsmtSF",
    "BsmtExposure": "TotalBsmtSF",
    "BsmtFinType1": "TotalBsmtSF",
    "BsmtFinType2": "TotalBsmtSF"
}

for column, evidence in feature_columns.items():
    is_absent = df[column].isna() & (df[evidence] == 0)
    df.loc[is_absent, column] = "None"
    df[column] = df[column].fillna("Unknown")

## 5. Fix genuine missing measurements and add missingness flags

In [ ]:
# Street frontage: use the median for the same neighborhood.
df["LotFrontage_missing"] = df["LotFrontage"].isna().astype(int)
local_median = df.groupby("Neighborhood")["LotFrontage"].transform("median")
df["LotFrontage"] = df["LotFrontage"].fillna(local_median)
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

# Electrical: use the most common category (mode).
df["Electrical_missing"] = df["Electrical"].isna().astype(int)
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# Masonry veneer: real "None" must not be confused with missing data.
df["MasVnrType_missing"] = df["MasVnrType"].isna().astype(int)
no_veneer = df["MasVnrType"].isna() & (df["MasVnrArea"] == 0)
df.loc[no_veneer, "MasVnrType"] = "None"
df["MasVnrType"] = df["MasVnrType"].fillna("Unknown")

df["MasVnrArea_missing"] = df["MasVnrArea"].isna().astype(int)
no_veneer_area = df["MasVnrArea"].isna() & (df["MasVnrType"] == "None")
df.loc[no_veneer_area, "MasVnrArea"] = 0
positive_area_median = df.loc[df["MasVnrArea"] > 0, "MasVnrArea"].median()
df["MasVnrArea"] = df["MasVnrArea"].fillna(positive_area_median)

# A house without a garage has no GarageYrBlt. Leave NA and add a flag.
df["GarageYrBlt_missing"] = df["GarageYrBlt"].isna().astype(int)

print("Remaining missing values:")
print(df.isna().sum()[df.isna().sum() > 0])

## 6. Correct column data types

In [ ]:
# These numbers represent labels or calendar months, not measurements.
df["MSSubClass"] = df["MSSubClass"].astype("str").astype("category")
df["MoSold"] = df["MoSold"].astype("category")

# Turn all descriptive text columns into categories.
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].astype("category")

# Garage construction years should be whole numbers, but missing is allowed.
df["GarageYrBlt"] = df["GarageYrBlt"].astype("Int64")

print(df[["MSSubClass", "MoSold", "MSZoning", "GarageYrBlt", "SalePrice"]].dtypes)

## 7. Identify three outlier columns using IQR (do not delete)

In [ ]:
outlier_results = []

for column in ["GrLivArea", "LotArea", "GarageArea"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    is_outlier = (df[column] < low) | (df[column] > high)
    df[column + "_outlier"] = is_outlier.astype(int)

    outlier_results.append({
        "Column": column,
        "Lower_Bound": round(low, 2),
        "Upper_Bound": round(high, 2),
        "Outlier_Count": is_outlier.sum(),
        "Decision": "Keep and flag; unusual does not prove an error"
    })

outlier_report = pd.DataFrame(outlier_results)
print(outlier_report.to_string(index=False))
outlier_report.to_csv("outlier_report.csv", index=False)

## 8. Final quality checks and save the cleaned CSV

In [ ]:
print("Original rows:", len(raw))
print("Cleaned rows:", len(df))
print("SalePrice unchanged:", df["SalePrice"].equals(original_price))
print("Id unchanged:", df["Id"].equals(original_ids))
print("Remaining missing cells:", df.isna().sum().sum())
print("Nonpositive LotArea values:", (df["LotArea"] <= 0).sum())
print("Nonpositive GrLivArea values:", (df["GrLivArea"] <= 0).sum())
print("Negative GarageArea values:", (df["GarageArea"] < 0).sum())

# Do not drop outliers, rows, columns, or change SalePrice.
df.to_csv("ames_housing_cleaned.csv", index=False, na_rep="NA")
pd.DataFrame({"Column": df.columns, "Data_Type": df.dtypes.astype(str).values}).to_csv(
    "data_types_report.csv", index=False
)

print("Finished. Cleaned data saved as ames_housing_cleaned.csv")